# Bronze - Data Profiling y Calidad Oficial

Notebook oficial de la capa Bronze. Integra las fuentes del proyecto municipal:

- SIAF presupuesto y ejecución de ingresos.
- SISMEPRE seguimiento de meta de impuesto predial.
- RENAMU 2022.
- `CategoriasMunicipalidades.csv`, entregado por el profesor.

Bronze no modela ni corrige reglas de negocio complejas. Su responsabilidad es aterrizar Raw como Parquet Snappy con trazabilidad y evidencia de calidad inicial.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, types as T

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if PROJECT_ROOT.name in {"bronze", "silver", "gold"}:
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

spark = (
    SparkSession.builder
    .appName("municipal-medallion-profiling")
    .config("spark.sql.parquet.mergeSchema", "true")
    .getOrCreate()
)

def path_exists(path: str) -> bool:
    return Path(path).exists()

def read_parquet(path: str):
    if not path_exists(path):
        print(f"No existe: {path}")
        return None
    return spark.read.parquet(path)

def show_df(df, n=10, truncate=False):
    if df is None:
        print("DataFrame no disponible")
    else:
        df.show(n, truncate=truncate)

def count_nulls_and_blanks(df):
    exprs = []
    for c, dtype in df.dtypes:
        if dtype == "string":
            exprs.append(F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c))
        else:
            exprs.append(F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c))
    return df.select(exprs)

def summarize_table(name: str, df, business_keys=None):
    business_keys = business_keys or []
    rows = df.count()
    cols = len(df.columns)
    duplicates = rows - df.dropDuplicates().count()
    print(f"Tabla: {name}")
    print(f"Registros: {rows:,}")
    print(f"Columnas: {cols}")
    print(f"Duplicados exactos: {duplicates:,}")
    if business_keys and all(c in df.columns for c in business_keys):
        dup_keys = df.groupBy(*business_keys).count().filter("count > 1").count()
        print(f"Duplicados por clave {business_keys}: {dup_keys:,}")
    df.printSchema()
    return {"table": name, "rows": rows, "columns": cols, "duplicates": duplicates}

def domain_check(df, column, valid_values):
    return (
        df.groupBy(column)
        .count()
        .withColumn("is_valid_domain", F.col(column).isin(list(valid_values)))
        .orderBy(F.desc("count"))
    )

## 1. Inventario Raw y Bronze

Objetivo: comprobar que las fuentes están presentes y que Bronze publicó Parquet por dataset. Esto demuestra almacenamiento columnar, reproducible y con trazabilidad.

In [ ]:
raw_root = PROJECT_ROOT / "data" / "raw"
bronze_root = PROJECT_ROOT / "data" / "bronze"

raw_inventory = []
for path in sorted(raw_root.rglob("*")):
    if path.is_file():
        raw_inventory.append((str(path.relative_to(PROJECT_ROOT)), path.stat().st_size))

bronze_inventory = []
for path in sorted(bronze_root.iterdir()) if bronze_root.exists() else []:
    if path.is_dir():
        parquet_count = len(list(path.rglob("*.parquet")))
        bronze_inventory.append((path.name, parquet_count, str(path.relative_to(PROJECT_ROOT))))

spark.createDataFrame(raw_inventory, ["raw_file", "bytes"]).show(100, truncate=False)
spark.createDataFrame(bronze_inventory, ["bronze_dataset", "parquet_files", "path"]).show(100, truncate=False)

## 2. Lectura de fuentes Bronze

Objetivo: cargar los Parquet Bronze y verificar disponibilidad. Si una fuente no aparece aquí, no debe pasar a Silver.

In [ ]:
bronze_tables = {
    "ingresos": read_parquet(str(bronze_root / "ingresos")),
    "sismepre_respuestas": read_parquet(str(bronze_root / "rentas_respuestas")),
    "sismepre_predial": read_parquet(str(bronze_root / "rentas_esat_estadistica_atm")),
    "sismepre_entidad_estado": read_parquet(str(bronze_root / "rentas_entidad_estado")),
    "sismepre_formularios": read_parquet(str(bronze_root / "rentas_formularios")),
    "sismepre_preguntas": read_parquet(str(bronze_root / "rentas_preguntas")),
    "renamu": read_parquet(str(bronze_root / "renamu")),
    "categorias_municipalidades": read_parquet(str(bronze_root / "categorias_municipalidades")),
}

[(name, df is not None) for name, df in bronze_tables.items()]

## 3. Conteos, esquemas y duplicados

Objetivo: medir volumen, estructura y duplicados exactos por fuente antes de cualquier limpieza. Impacto: fija la línea base de auditoría para explicar cambios en Silver y Gold.

In [ ]:
bronze_summary = []
business_keys = {
    "ingresos": ["ANO_DOC", "MES_DOC", "SEC_EJEC", "EJECUTORA"],
    "sismepre_predial": ["SEC_EJEC", "ANO_APLICACION", "PERIODO", "ANO_ESTADISTICA", "MES_ESTADISTICA", "FORMULARIO_ID"],
    "sismepre_respuestas": ["SEC_EJEC", "ANO_APLICACION", "PERIODO", "FORMULARIO_ID", "PREGUNTA_ID", "RESPUESTA_ID"],
    "categorias_municipalidades": ["Municipalidad", "Categoria"],
}

for name, df in bronze_tables.items():
    if df is None:
        continue
    bronze_summary.append(summarize_table(name, df, business_keys.get(name, [])))

spark.createDataFrame(bronze_summary).show(truncate=False)

## 4. Trazabilidad Bronze

Objetivo: validar columnas `_bronze_*`. Si falta trazabilidad, no se puede explicar origen, checksum, ejecución ni fecha de ingesta.

In [ ]:
trace_cols = ["_bronze_source_path", "_bronze_source_url", "_bronze_source_checksum", "_bronze_execution_id", "_bronze_ingestion_ts", "_bronze_ingestion_date"]
trace_rows = []
for name, df in bronze_tables.items():
    if df is None:
        continue
    cols = set(df.columns)
    missing = [c for c in trace_cols if c not in cols]
    trace_rows.append((name, len(missing) == 0, ",".join(missing)))
spark.createDataFrame(trace_rows, ["dataset", "trace_ok", "missing_trace_columns"]).show(truncate=False)

## 5. Nulos, vacíos y espacios

Objetivo: detectar campos incompletos, cadenas vacías y espacios en blanco. Corrección recomendada: Silver debe normalizar strings con `trim`, convertir vacíos a null y tipar columnas críticas.

In [ ]:
for name, df in bronze_tables.items():
    if df is None:
        continue
    print(f"\n=== {name} ===")
    count_nulls_and_blanks(df).show(truncate=False)

## 6. Distribuciones y dominios principales

Objetivo: revisar valores dominantes y fuera de dominio. En Bronze se reporta; en Silver se corrige o cuarentena.

In [ ]:
if bronze_tables["ingresos"] is not None and "NIVEL_GOBIERNO" in bronze_tables["ingresos"].columns:
    bronze_tables["ingresos"].groupBy("NIVEL_GOBIERNO").count().orderBy(F.desc("count")).show()

if bronze_tables["categorias_municipalidades"] is not None:
    domain_check(bronze_tables["categorias_municipalidades"], "Categoria", list("ABCDEFG")).show(50, truncate=False)

for name in ["sismepre_entidad_estado", "sismepre_predial", "renamu"]:
    df = bronze_tables.get(name)
    if df is not None:
        print(f"\nColumnas muestra {name}:")
        df.select(df.columns[: min(8, len(df.columns))]).show(5, truncate=False)

## 7. Validación de particiones y Parquet

Objetivo: evidenciar que Bronze está en Parquet y que las fuentes temporales usan particiones cuando corresponde.

In [ ]:
partition_rows = []
for table_dir in sorted(bronze_root.iterdir()) if bronze_root.exists() else []:
    if not table_dir.is_dir():
        continue
    partitions = sorted({p.parent.name for p in table_dir.rglob("*.parquet") if "=" in p.parent.name})
    partition_rows.append((table_dir.name, len(list(table_dir.rglob("*.parquet"))), ",".join(partitions[:10])))
spark.createDataFrame(partition_rows, ["dataset", "parquet_files", "sample_partitions"]).show(200, truncate=False)

## 8. Calidad Bronze - ocho dimensiones

Objetivo: documentar la evaluación inicial con completitud, unicidad, validez, consistencia, integridad, actualidad, disponibilidad y exactitud.

In [ ]:
quality_rows = []
for name, df in bronze_tables.items():
    available = df is not None
    if not available:
        quality_rows.append((name, "disponibilidad", 0.0, "Fuente Bronze no disponible"))
        continue
    total = df.count()
    exact_unique = df.dropDuplicates().count()
    trace_ok = all(c in df.columns for c in trace_cols)
    quality_rows.extend([
        (name, "disponibilidad", 100.0, "Parquet disponible"),
        (name, "unicidad", round(exact_unique / total * 100, 2) if total else 0.0, "Duplicados exactos medidos"),
        (name, "trazabilidad", 100.0 if trace_ok else 0.0, "Columnas _bronze_*"),
        (name, "actualidad", 100.0 if "_bronze_ingestion_ts" in df.columns else 0.0, "Fecha de ingesta disponible"),
    ])
spark.createDataFrame(quality_rows, ["dataset", "dimension", "score_pct", "interpretacion"]).show(200, truncate=False)

## 9. Conclusión Bronze

Bronze queda como evidencia de aterrizaje: Raw convertido a Parquet, trazabilidad, conteos, esquemas y alertas iniciales. Las correcciones de tipos, nulos, duplicados y categorías se realizan en Silver.